## **Question 1. Consider the following multi-agent problem** 

### **(a) Scenario** 

i. Number of agents: 2 agents 

ii. Grid world size: 5 + (last digit of your roll number) 

iii. Both agents must reach the goal simultaneously to receive reward = +10 

iv. Otherwise reward = 0 

v. Episode length = 30 steps 

### **(b) Based on this environment implement Independent Q-learning for both agents.** 

i. Train for last 4 digits of your roll number episodes and submit learning curve (average reward vs episodes) 

ii. Grid world size: 5 + (last digit of your roll number) 

iii. Use ε-greedy exploration with decay 

iv. Identify one non-trivial pattern in your learning curve 

v. Episode length = 30 steps 

### **(c) Modify one aspect** 

Learning rate (if Γ = 0, 5, 9)

ε schedule (if Γ = 2, 4, 8)

Reward shaping (otherwise) 


• Show new learning curve 

• Explain what changed and why 

---

# Importing Libraries

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import random
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

In [3]:

import random
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("Libraries imported successfully.")

In [4]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# Grid World Environment

In [5]:
class CoopGridWorld:

    UP    = 0
    DOWN  = 1
    LEFT  = 2
    RIGHT = 3
    STAY = 4
    N_ACTIONS = 5

    ACTION_NAMES = {0: '↑', 1: '↓', 2: '←', 3: '→', 4: '•'}

    def __init__(self, grid_size=10, max_steps=30, goal=None, start1=None, start2=None):
        self.grid_size  = grid_size
        self.max_steps  = max_steps

        self.goal = goal if goal else (grid_size - 1, grid_size - 1)

        self.start1 = start1 if start1 else (0, 0)
        self.start2 = start2 if start2 else (0, grid_size - 1)

        self.reset()

    def reset(self):
        self.pos1     = list(self.start1)
        self.pos2     = list(self.start2)
        self.step_count = 0
        return self._get_obs()

    def _get_obs(self):
        obs1 = tuple(self.pos1)
        obs2 = tuple(self.pos2)
        return obs1, obs2

    def _move(self, pos, action):
        row, col = pos
        if action == self.UP:    row -= 1
        elif action == self.DOWN: row += 1
        elif action == self.LEFT: col -= 1
        elif action == self.RIGHT: col += 1
        elif action == self.STAY: pass
        
        row = max(0, min(self.grid_size - 1, row))
        col = max(0, min(self.grid_size - 1, col))

        return [row, col]

    def step(self, action1, action2, reward_shaping=False, q1=None, q2=None):
        
        self.step_count += 1

        self.pos1 = self._move(self.pos1, action1)
        self.pos2 = self._move(self.pos2, action2)

        at_goal1 = tuple(self.pos1) == self.goal
        at_goal2 = tuple(self.pos2) == self.goal

        if at_goal1 and at_goal2:
            reward = 10.0
            done   = True
        else:
            reward = 0.0
            done   = (self.step_count >= self.max_steps)

        obs1, obs2 = self._get_obs()
        info = {'at_goal1': at_goal1, 'at_goal2': at_goal2, 'steps': self.step_count}
        return obs1, obs2, reward, done, info

    def render(self, episode=None):
        
        grid = [['.' for _ in range(self.grid_size)]
                for _ in range(self.grid_size)]
        gr, gc = self.goal
        grid[gr][gc] = 'G'
        r2, c2 = self.pos2
        r1, c1 = self.pos1
        grid[r2][c2] = '2'
        grid[r1][c1] = '1'
        header = f"Episode {episode} | Step {self.step_count}" if episode else f"Step {self.step_count}"
        print(header)
        for row in grid:
            print(' '.join(row))
        print()

last_digit = 5
grid_size = 5 + last_digit

env_test = CoopGridWorld(grid_size=grid_size, max_steps=30)
o1, o2 = env_test.reset()
print(f"Environment created  →  Grid: {5+last_digit}×{5+last_digit} | Goal: {env_test.goal}")
print(f"   Agent 1 starts at: {o1}")
print(f"   Agent 2 starts at: {o2}")
env_test.render()

# Independent Q-Learning Agent

In [6]:
class IQLAgent:
    def __init__(self, agent_id, grid_size, n_actions, alpha=0.1, gamma=0.95, eps_start=1.0, eps_min=0.05, eps_decay=0.995):

        self.agent_id  = agent_id
        self.grid_size = grid_size
        self.n_actions = n_actions

        self.alpha     = alpha
        self.gamma     = gamma
        self.epsilon   = eps_start
        self.eps_min   = eps_min
        self.eps_decay = eps_decay

        self.Q = defaultdict(lambda: np.zeros(n_actions))

        self.episode_count = 0
        self.epsilon_history = []

    # def select_action(self, state):

    #     if np.random.rand() < self.epsilon:
    #         return np.random.randint(self.n_actions)
    #     else:
    #         return np.argmax(self.Q[state])

    def select_action(self, state):
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)
        else:
            q_vals = self.Q[state]
            max_q = np.max(q_vals)
            actions = np.where(q_vals == max_q)[0]
            return np.random.choice(actions)

    def update(self, state, action, reward, next_state, done):
        current_q = self.Q[state][action]

        if done:
            td_target = reward
        else:
            td_target = reward + self.gamma * np.max(self.Q[next_state])

        td_error = td_target - current_q
        self.Q[state][action] += self.alpha * td_error

    def decay_epsilon(self):
        self.episode_count += 1
        self.epsilon = max(self.eps_min, self.epsilon * self.eps_decay)
        self.epsilon_history.append(self.epsilon)

    def get_policy_grid(self):
        policy = np.zeros((self.grid_size, self.grid_size), dtype=int)
        for r in range(self.grid_size):
            for c in range(self.grid_size):
                # policy[r, c] = np.argmax(self.Q[(r, c)])
                q_vals = self.Q[(r, c)]
                max_q = np.max(q_vals)
                actions = np.where(q_vals == max_q)[0]
                policy[r, c] = np.random.choice(actions)

        return policy

    def get_value_grid(self):
        values = np.zeros((self.grid_size, self.grid_size))
        for r in range(self.grid_size):
            for c in range(self.grid_size):
                values[r, c] = np.max(self.Q[(r, c)])
        return values

test_agent = IQLAgent('Test', grid_size=grid_size, n_actions=5)
state = (0, 0)
action = test_agent.select_action(state)
print(f" IQLAgent created")
print(f"   Test action from (0,0): {CoopGridWorld.ACTION_NAMES[action]}")
print(f"   Initial epsilon: {test_agent.epsilon}")
print(f"   Q-table size: {len(test_agent.Q)} states visited so far")

## Training Loop (Independent Q-Learning Baseline)

In [7]:
def train_iql(env, agent1, agent2, n_episodes, verbose_every=200, reward_shaping=False):
              
    episode_rewards = []
    success_flags   = []

    for ep in range(1, n_episodes + 1):
        obs1, obs2 = env.reset()
        total_reward = 0.0
        done = False

        def phi(pos):
            gr, gc = env.goal
            return -(abs(pos[0] - gr) + abs(pos[1] - gc))

        while not done:
            a1 = agent1.select_action(obs1)
            a2 = agent2.select_action(obs2)

            old_obs1, old_obs2 = obs1, obs2

            obs1, obs2, reward, done, info = env.step(a1, a2)

            if reward_shaping:
                shaping1 = agent1.gamma * phi(obs1) - phi(old_obs1)
                shaping2 = agent2.gamma * phi(obs2) - phi(old_obs2)
                reward1 = reward + shaping1
                reward2 = reward + shaping2
            else:
                reward1 = reward
                reward2 = reward

            agent1.update(old_obs1, a1, reward1, obs1, done)
            agent2.update(old_obs2, a2, reward2, obs2, done)

            total_reward += reward

        agent1.decay_epsilon()
        agent2.decay_epsilon()

        episode_rewards.append(total_reward)
        success_flags.append(total_reward > 0)

        if ep % verbose_every == 0 or ep == 1:
            recent_sr = np.mean(success_flags[-verbose_every:]) * 100
            print(f"  Ep {ep:5d}/{n_episodes} | "
                  f"ε={agent1.epsilon:.3f} | "
                  f"Success Rate (last {verbose_every}): {recent_sr:.1f}%")

    return episode_rewards, success_flags


print("Training function has been defined")

In [8]:
last_digit = 5
last_four_digit = 2185
GRID_SIZE   = 5 + last_digit
N_EPISODES  = last_four_digit
MAX_STEPS   = 30
ALPHA       = 0.1
GAMMA       = 0.95
EPS_START   = 1.0
EPS_MIN     = 0.05
EPS_DECAY   = 0.990

print("    BASELINE: Independent Q-Learning\n")

print(f"  Grid Size   : {GRID_SIZE} × {GRID_SIZE}")
print(f"  Episodes    : {N_EPISODES}")
print(f"  Max Steps   : {MAX_STEPS}")
print(f"  Alpha (α)   : {ALPHA}")
print(f"  Gamma (γ)   : {GAMMA}")
print(f"  ε start     : {EPS_START}")
print(f"  ε min       : {EPS_MIN}")
print(f"  ε decay     : {EPS_DECAY}")

env_base = CoopGridWorld(grid_size=GRID_SIZE, max_steps=MAX_STEPS)

agent1_base = IQLAgent('Agent1', GRID_SIZE, CoopGridWorld.N_ACTIONS,
                       alpha=ALPHA, gamma=GAMMA,
                       eps_start=EPS_START, eps_min=EPS_MIN,
                       eps_decay=EPS_DECAY)
agent2_base = IQLAgent('Agent2', GRID_SIZE, CoopGridWorld.N_ACTIONS,
                       alpha=ALPHA, gamma=GAMMA,
                       eps_start=EPS_START, eps_min=EPS_MIN,
                       eps_decay=EPS_DECAY)

print("\n Training started...\n")
rewards_base, success_base = train_iql(
    env_base, agent1_base, agent2_base,
    n_episodes=N_EPISODES,
    verbose_every=200,
    reward_shaping=False
)

print(f"\nTraining complete!")
print(f"   Overall success rate: {np.mean(success_base)*100:.2f}%")
print(f"   Final ε (Agent 1): {agent1_base.epsilon:.4f}")
print(f"   Q-table entries (A1): {len(agent1_base.Q)} | (A2): {len(agent2_base.Q)}")

## Learning Curve (Baseline Independent Q-Learning)

In [9]:
def smooth(data, window=100):
    return np.convolve(data, np.ones(window)/window, mode='valid')

def plot_learning_curve(rewards, title, color='royalblue', window=100, ax=None, label=None):
    episodes = np.arange(1, len(rewards) + 1)
    smoothed = smooth(rewards, window)
    ep_smooth = episodes[window-1:]

    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 5))

    ax.plot(episodes, rewards, alpha=0.15, color=color, linewidth=0.5)
    
    lbl = label if label else f'Rolling Avg (window={window})'
    ax.plot(ep_smooth, smoothed, color=color, linewidth=2, label=lbl)


    ax.axhline(y=10, color='green', linestyle='--', alpha=0.6, label='Max Reward = 10')

    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Episode', fontsize=11)
    ax.set_ylabel('Average Reward', fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(1, len(rewards))
    ax.set_ylim(-0.5, 11)

    return ax


fig, axes = plt.subplots(2, 1, figsize=(13, 10))

plot_learning_curve(rewards_base, title='Baseline IQL: Average Reward vs Episodes', color='royalblue', ax=axes[0])

chunk = 100
n_chunks = len(success_base) // chunk
chunk_sr = [np.mean(success_base[i*chunk:(i+1)*chunk]) * 10
            for i in range(n_chunks)]
axes[1].bar(np.arange(n_chunks) * chunk + chunk//2, chunk_sr,
            width=chunk*0.85, color='steelblue', alpha=0.7)
axes[1].axhline(y=10, color='green', linestyle='--', alpha=0.6, label='Max = 10')
axes[1].set_title('Success Rate per 100-Episode Block (scaled to reward space)', fontsize=12)
axes[1].set_xlabel('Episode', fontsize=11)
axes[1].set_ylabel('Avg Reward in Block', fontsize=11)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('Q1_baseline_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved as 'Q1_baseline_learning_curve.png'")

A key non-trivial pattern observed is that the average reward remains at zero throughout training despite a steady decay in exploration rate. Specifically, epsilon reaches its minimum value early (around episode 600), after which exploration significantly reduces. Since no successful coordinated trajectory is discovered during the high-exploration phase, the agents converge prematurely to suboptimal behavior with near-zero Q-values. This highlights the difficulty of learning in sparse-reward multi-agent environments, where coordination must emerge through rare joint exploration.

# Reward-Shaped Indpendent Q-Learning Training

In [10]:
print("    MODIFIED: IQL + Reward Shaping\n")
print("  Modification : Potential-Based Reward Shaping")
print("  Φ(s)         : -Manhattan distance to goal")
print("  F(s, s')     : γ·Φ(s') - Φ(s)")
print("  All other hyperparameters: SAME as baseline\n")


np.random.seed(SEED)
random.seed(SEED)

env_shaped = CoopGridWorld(grid_size=GRID_SIZE, max_steps=MAX_STEPS)

agent1_shaped = IQLAgent('Agent1_S', GRID_SIZE, CoopGridWorld.N_ACTIONS,
                         alpha=ALPHA, gamma=GAMMA,
                         eps_start=EPS_START, eps_min=EPS_MIN,
                         eps_decay=EPS_DECAY)
agent2_shaped = IQLAgent('Agent2_S', GRID_SIZE, CoopGridWorld.N_ACTIONS,
                         alpha=ALPHA, gamma=GAMMA,
                         eps_start=EPS_START, eps_min=EPS_MIN,
                         eps_decay=EPS_DECAY)

print("\nTraining started...\n")
rewards_shaped, success_shaped = train_iql(
    env_shaped, agent1_shaped, agent2_shaped,
    n_episodes=N_EPISODES,
    verbose_every=200,
    reward_shaping=True
)

print(f"\nTraining complete!")
print(f"   Overall success rate (shaped): {np.mean(success_shaped)*100:.2f}%")

# Learning Curve (Independent Q-Learning + Reward Shaping)

In [11]:
fig, axes = plt.subplots(2, 1, figsize=(13, 10))

episodes = np.arange(1, N_EPISODES + 1)
window = 100

sm_shaped = smooth(rewards_shaped, window)
ep_sm = episodes[window-1:]

ax = axes[0]

ax.plot(episodes, rewards_shaped, alpha=0.08, color='darkorange', lw=0.5)
ax.plot(ep_sm, sm_shaped, color='darkorange', lw=2.2, label='IQL + Reward Shaping')

ax.axhline(y=10, color='green', linestyle='--', alpha=0.5, lw=1.5, label='Max Reward = 10')

ax.set_title('IQL + Reward Shaping: Average Reward vs Episodes', fontsize=13, fontweight='bold')
ax.set_xlabel('Episode', fontsize=11)
ax.set_ylabel('Rolling Avg Reward (window=100)', fontsize=11)

ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(1, N_EPISODES)
ax.set_ylim(-0.5, 11)

ax2 = axes[1]

success_block = []
block_size = 100

for i in range(0, len(success_shaped), block_size):
    block = success_shaped[i:i+block_size]
    success_block.append(np.mean(block) * 10)

episodes_block = np.arange(len(success_block)) * block_size

ax2.plot(episodes_block, success_block, color='darkorange', lw=2)

ax2.axhline(10, color='green', linestyle='--', alpha=0.5, label='Max = 10')

ax2.set_title('Success Rate per 100 Episodes (Scaled)', fontsize=12)
ax2.set_xlabel('Episode', fontsize=11)
ax2.set_ylabel('Avg Reward in Block', fontsize=11)

ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.savefig('Q1_shaped_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print("Figure saved as 'Q1_shaped_learning_curve.png'")

# Comparison Plot (Baseline vs Reward Shaped)

In [12]:
fig, axes = plt.subplots(2, 1, figsize=(13, 10))

ax = axes[0]
episodes = np.arange(1, N_EPISODES + 1)
window = 100

sm_base   = smooth(rewards_base,   window)
sm_shaped = smooth(rewards_shaped, window)
ep_sm = episodes[window-1:]

ax.plot(episodes, rewards_base,   alpha=0.08, color='royalblue', lw=0.5)
ax.plot(episodes, rewards_shaped, alpha=0.08, color='darkorange', lw=0.5)
ax.plot(ep_sm, sm_base,   color='royalblue',  lw=2.2, label='Baseline IQL')
ax.plot(ep_sm, sm_shaped, color='darkorange', lw=2.2, label='IQL + Reward Shaping')
ax.axhline(y=10, color='green', linestyle='--', alpha=0.5, lw=1.5, label='Max Reward = 10')

ax.set_title('IQL: Baseline vs Reward Shaping', fontsize=13, fontweight='bold')
ax.set_xlabel('Episode', fontsize=11)
ax.set_ylabel('Rolling Avg Reward (window=100)', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(1, N_EPISODES)
ax.set_ylim(-0.5, 11)

half = N_EPISODES // 2
base_first_half  = np.mean(sm_base[:half//1])
shape_first_half = np.mean(sm_shaped[:half//1])
if shape_first_half > base_first_half:
    ax.annotate('Shaped learns\nfaster early on',
                xy=(half//3, shape_first_half + 0.3),
                fontsize=9, color='darkorange',
                ha='center')

ax2 = axes[1]
diff = sm_shaped - sm_base
colors = ['darkorange' if d > 0 else 'royalblue' for d in diff]
ax2.bar(ep_sm, diff, color=colors, alpha=0.6, width=1)
ax2.axhline(0, color='black', lw=1)
ax2.set_title('Δ Reward (Shaped − Baseline): Positive = Shaping Helps', fontsize=12)
ax2.set_xlabel('Episode', fontsize=11)
ax2.set_ylabel('Δ Average Reward', fontsize=11)
ax2.grid(True, alpha=0.3)

orange_patch = mpatches.Patch(color='darkorange', label='Shaping better')
blue_patch   = mpatches.Patch(color='royalblue',  label='Baseline better')
ax2.legend(handles=[orange_patch, blue_patch], fontsize=10)

plt.tight_layout()
plt.savefig('Q1_comparison_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved as 'Q1_comparison_learning_curve.png'")

Reward shaping adds intermediate feedback based on distance to the goal, allowing agents to learn directional movement. This overcomes the sparse reward issue in the baseline, leading to faster learning, improved coordination, and significantly higher success rates.

## **2. Communication Constraint Problem (20 Marks)** 

Assume agents can send a 1-bit message (0 or 1) per timestep. Based on this assumption do the following: 

### **(a) Propose a communication strategy** 

• What does the 1 bit represent? 
• When is it sent? 

### **(b) Implement and show results** 

• Incorporate communication into learning 
• Train and compare with Problem 1 

### **(c) Submit learning curves** 

• With communication 
• Without communication 

### **(d) Answer the following** 

i. Did communication help? Why or why not? 
ii. Identify one failure case of your communication scheme 
iii. What is the information bottleneck? 

--- 


In [13]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

last_digit       = 5
last_four_digit  = 2185

GRID_SIZE   = 5 + last_digit
N_EPISODES  = last_four_digit
MAX_STEPS   = 30
ALPHA       = 0.1
GAMMA       = 0.95
EPS_START   = 1.0
EPS_MIN     = 0.05
EPS_DECAY   = 0.995
COMM_THRESH = 3

# Grid World (reused from Q1)

In [14]:
class CoopGridWorld:
    UP, DOWN, LEFT, RIGHT, STAY = 0, 1, 2, 3, 4
    N_ACTIONS = 5
    ACTION_NAMES = {0:'↑', 1:'↓', 2:'←', 3:'→', 4:'•'}

    def __init__(self, grid_size=10, max_steps=30, goal=None,
                 start1=None, start2=None):
        self.grid_size  = grid_size
        self.max_steps  = max_steps
        self.goal       = goal   if goal   else (grid_size-1, grid_size-1)
        self.start1     = start1 if start1 else (0, 0)
        self.start2     = start2 if start2 else (0, grid_size-1)
        self.reset()

    def reset(self):
        self.pos1 = list(self.start1)
        self.pos2 = list(self.start2)
        self.step_count = 0
        return self._get_obs()

    def _get_obs(self):
        return tuple(self.pos1), tuple(self.pos2)

    def _move(self, pos, action):
        r, c = pos
        if   action == self.UP:    r -= 1
        elif action == self.DOWN:  r += 1
        elif action == self.LEFT:  c -= 1
        elif action == self.RIGHT: c += 1
        r = max(0, min(self.grid_size-1, r))
        c = max(0, min(self.grid_size-1, c))
        return [r, c]

    def step(self, action1, action2):
        self.step_count += 1
        self.pos1 = self._move(self.pos1, action1)
        self.pos2 = self._move(self.pos2, action2)
        at_goal1 = tuple(self.pos1) == self.goal
        at_goal2 = tuple(self.pos2) == self.goal
        if at_goal1 and at_goal2:
            reward, done = 10.0, True
        else:
            reward = 0.0
            done   = (self.step_count >= self.max_steps)
        return self._get_obs()[0], self._get_obs()[1], reward, done, {}

# IQL Agent (Extended state = (row, col, partner_bit))

In [15]:
class IQLAgentComm:
    def __init__(self, agent_id, grid_size, n_actions,
                 alpha=0.1, gamma=0.95,
                 eps_start=1.0, eps_min=0.05, eps_decay=0.995):
        self.agent_id  = agent_id
        self.n_actions = n_actions
        self.alpha     = alpha
        self.gamma     = gamma
        self.epsilon   = eps_start
        self.eps_min   = eps_min
        self.eps_decay = eps_decay
        self.Q         = defaultdict(lambda: np.zeros(n_actions))

    def _augment(self, pos, partner_bit):
        return (pos[0], pos[1], int(partner_bit))

    def select_action(self, pos, partner_bit):
        state = self._augment(pos, partner_bit)
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)
        q = self.Q[state]
        return int(np.random.choice(np.where(q == q.max())[0]))

    def update(self, pos, partner_bit, action, reward,
               next_pos, next_partner_bit, done):
        s  = self._augment(pos, partner_bit)
        s_ = self._augment(next_pos, next_partner_bit)
        target = reward if done else reward + self.gamma * self.Q[s_].max()
        self.Q[s][action] += self.alpha * (target - self.Q[s][action])

    def decay_epsilon(self):
        self.epsilon = max(self.eps_min, self.epsilon * self.eps_decay)

def compute_comm_bit(pos, goal, threshold=COMM_THRESH):
    return int(abs(pos[0]-goal[0]) + abs(pos[1]-goal[1]) <= threshold)

# Training loop (IQL + Reward Shaping + Communication)

In [16]:
def train_iql_comm(env, agent1, agent2, n_episodes,
                   verbose_every=200):
    episode_rewards = []
    success_flags   = []

    def phi(pos):
        gr, gc = env.goal
        return -(abs(pos[0]-gr) + abs(pos[1]-gc))

    for ep in range(1, n_episodes+1):
        obs1, obs2 = env.reset()
        total_reward = 0.0
        done = False

        bit1 = compute_comm_bit(obs1, env.goal)
        bit2 = compute_comm_bit(obs2, env.goal)

        while not done:
            a1 = agent1.select_action(obs1, partner_bit=bit2)
            a2 = agent2.select_action(obs2, partner_bit=bit1)

            old_obs1, old_obs2 = obs1, obs2
            old_bit1, old_bit2 = bit1, bit2

            obs1, obs2, reward, done, _ = env.step(a1, a2)

            bit1 = compute_comm_bit(obs1, env.goal)
            bit2 = compute_comm_bit(obs2, env.goal)

            shaping1 = GAMMA * phi(obs1) - phi(old_obs1)
            shaping2 = GAMMA * phi(obs2) - phi(old_obs2)
            r1 = reward + shaping1
            r2 = reward + shaping2

            agent1.update(old_obs1, old_bit2, a1, r1, obs1, bit2, done)
            agent2.update(old_obs2, old_bit1, a2, r2, obs2, bit1, done)

            total_reward += reward

        agent1.decay_epsilon()
        agent2.decay_epsilon()
        episode_rewards.append(total_reward)
        success_flags.append(total_reward > 0)

        if ep % verbose_every == 0 or ep == 1:
            recent_sr = np.mean(success_flags[-verbose_every:]) * 100
            print(f"  Ep {ep:5d}/{n_episodes} | "
                  f"ε={agent1.epsilon:.3f} | "
                  f"Success (last {verbose_every}): {recent_sr:.1f}%")

    return episode_rewards, success_flags

# IQL + Reward Shaping (no communication, replicate Q1 result)

In [17]:
np.random.seed(SEED); random.seed(SEED)

class IQLAgent:
    def __init__(self, agent_id, grid_size, n_actions,
                 alpha=0.1, gamma=0.95,
                 eps_start=1.0, eps_min=0.05, eps_decay=0.995):
        self.agent_id  = agent_id
        self.n_actions = n_actions
        self.alpha     = alpha
        self.gamma     = gamma
        self.epsilon   = eps_start
        self.eps_min   = eps_min
        self.eps_decay = eps_decay
        self.Q         = defaultdict(lambda: np.zeros(n_actions))

    def select_action(self, state):
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)
        q = self.Q[state]
        return int(np.random.choice(np.where(q == q.max())[0]))

    def update(self, s, a, r, s_, done):
        target = r if done else r + self.gamma * self.Q[s_].max()
        self.Q[s][a] += self.alpha * (target - self.Q[s][a])

    def decay_epsilon(self):
        self.epsilon = max(self.eps_min, self.epsilon * self.eps_decay)


def train_iql_shaped(env, agent1, agent2, n_episodes, verbose_every=200):
    episode_rewards, success_flags = [], []
    def phi(pos):
        gr, gc = env.goal
        return -(abs(pos[0]-gr) + abs(pos[1]-gc))
    for ep in range(1, n_episodes+1):
        obs1, obs2 = env.reset()
        total_reward = 0.0; done = False
        while not done:
            a1 = agent1.select_action(obs1)
            a2 = agent2.select_action(obs2)
            old1, old2 = obs1, obs2
            obs1, obs2, reward, done, _ = env.step(a1, a2)
            r1 = reward + GAMMA*phi(obs1) - phi(old1)
            r2 = reward + GAMMA*phi(obs2) - phi(old2)
            agent1.update(old1, a1, r1, obs1, done)
            agent2.update(old2, a2, r2, obs2, done)
            total_reward += reward
        agent1.decay_epsilon(); agent2.decay_epsilon()
        episode_rewards.append(total_reward)
        success_flags.append(total_reward > 0)
        if ep % verbose_every == 0 or ep == 1:
            sr = np.mean(success_flags[-verbose_every:])*100
            print(f"  Ep {ep:5d}/{n_episodes} | ε={agent1.epsilon:.3f} | "
                  f"Success (last {verbose_every}): {sr:.1f}%")
    return episode_rewards, success_flags

print("  IQL + Reward Shaping  (NO Communication)")
np.random.seed(SEED); random.seed(SEED)
env_nc   = CoopGridWorld(grid_size=GRID_SIZE, max_steps=MAX_STEPS)
a1_nc    = IQLAgent('A1', GRID_SIZE, 5, alpha=ALPHA, gamma=GAMMA,
                    eps_start=EPS_START, eps_min=EPS_MIN, eps_decay=EPS_DECAY)
a2_nc    = IQLAgent('A2', GRID_SIZE, 5, alpha=ALPHA, gamma=GAMMA,
                    eps_start=EPS_START, eps_min=EPS_MIN, eps_decay=EPS_DECAY)
rewards_nc, success_nc = train_iql_shaped(
    env_nc, a1_nc, a2_nc, N_EPISODES, verbose_every=200)
print(f"\nDone | Overall SR: {np.mean(success_nc)*100:.2f}%")

In [18]:
print("  IQL + Reward Shaping  (+1-bit Communication)")
np.random.seed(SEED); random.seed(SEED)
env_c  = CoopGridWorld(grid_size=GRID_SIZE, max_steps=MAX_STEPS)
a1_c   = IQLAgentComm('A1c', GRID_SIZE, 5, alpha=ALPHA, gamma=GAMMA,
                       eps_start=EPS_START, eps_min=EPS_MIN, eps_decay=EPS_DECAY)
a2_c   = IQLAgentComm('A2c', GRID_SIZE, 5, alpha=ALPHA, gamma=GAMMA,
                       eps_start=EPS_START, eps_min=EPS_MIN, eps_decay=EPS_DECAY)
rewards_c, success_c = train_iql_comm(
    env_c, a1_c, a2_c, N_EPISODES, verbose_every=200)
print(f"\nDone | Overall SR: {np.mean(success_c)*100:.2f}%")

# Learning Curves (Communication vs No Communication)

In [19]:
def smooth(data, w=100):
    return np.convolve(data, np.ones(w)/w, mode='valid')

WINDOW   = 100
episodes = np.arange(1, N_EPISODES+1)
ep_sm    = episodes[WINDOW-1:]
sm_nc    = smooth(rewards_nc,  WINDOW)
sm_c     = smooth(rewards_c,   WINDOW)

fig, axes = plt.subplots(3, 1, figsize=(13, 14))

ax = axes[0]
ax.plot(episodes, rewards_nc, alpha=0.07, color='darkorange', lw=0.5)
ax.plot(episodes, rewards_c,  alpha=0.07, color='mediumseagreen', lw=0.5)
ax.plot(ep_sm, sm_nc, color='darkorange',     lw=2.2,
        label='IQL + Shaped (no comm)')
ax.plot(ep_sm, sm_c,  color='mediumseagreen', lw=2.2,
        label='IQL + Shaped + 1-bit Comm')
ax.axhline(10, color='green', ls='--', alpha=0.5, lw=1.4, label='Max Reward = 10')
ax.set_title('Learning Curve: With vs Without 1-bit Communication',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Episode', fontsize=11)
ax.set_ylabel(f'Rolling Avg Reward (w={WINDOW})', fontsize=11)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
ax.set_xlim(1, N_EPISODES); ax.set_ylim(-0.5, 11)

ax2 = axes[1]
block = 100
n_blocks = N_EPISODES // block
sr_nc = [np.mean(success_nc[i*block:(i+1)*block])*10 for i in range(n_blocks)]
sr_c  = [np.mean(success_c [i*block:(i+1)*block])*10 for i in range(n_blocks)]
x_b   = np.arange(n_blocks) * block + block//2

ax2.plot(x_b, sr_nc, color='darkorange',     lw=2, label='No Comm')
ax2.plot(x_b, sr_c,  color='mediumseagreen', lw=2, label='1-bit Comm')
ax2.axhline(10, color='green', ls='--', alpha=0.5, label='Max = 10')
ax2.set_title('Success Rate per 100-Episode Block (scaled ×10)', fontsize=12)
ax2.set_xlabel('Episode', fontsize=11)
ax2.set_ylabel('Avg Reward in Block', fontsize=11)
ax2.legend(fontsize=10); ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, N_EPISODES)

ax3 = axes[2]
diff   = sm_c - sm_nc
colors = ['mediumseagreen' if d >= 0 else 'darkorange' for d in diff]
ax3.bar(ep_sm, diff, color=colors, alpha=0.65, width=1)
ax3.axhline(0, color='black', lw=1.2)
ax3.set_title('Δ Reward (Comm − No Comm): Green = Comm Helps', fontsize=12)
ax3.set_xlabel('Episode', fontsize=11)
ax3.set_ylabel('Δ Average Reward', fontsize=11)
ax3.grid(True, alpha=0.3)
g_patch = mpatches.Patch(color='mediumseagreen', label='Comm better')
o_patch = mpatches.Patch(color='darkorange',     label='No-Comm better')
ax3.legend(handles=[g_patch, o_patch], fontsize=10)

plt.tight_layout()
plt.savefig('Q2_comm_vs_nocomm_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → Q2_comm_vs_nocomm_learning_curve.png")

In [ ]:
final_100_nc = np.mean(success_nc[-100:]) * 100
final_100_c  = np.mean(success_c [-100:]) * 100
overall_nc   = np.mean(success_nc) * 100
overall_c    = np.mean(success_c ) * 100

print("           SUMMARY STATISTICS")
print(f"{'Metric':<35} {'No Comm':>9} {'1-bit Comm':>11}")
print(f"{'Overall success rate (%)' :<35} {overall_nc:>9.2f} {overall_c:>11.2f}")
print(f"{'Final-100-ep success rate (%)':<35} {final_100_nc:>9.2f} {final_100_c:>11.2f}")
print(f"{'Peak rolling avg reward':<35} {sm_nc.max():>9.3f} {sm_c.max():>11.3f}")
print(f"{'Q-table entries A1':<35} {len(a1_nc.Q):>9} {len(a1_c.Q):>11}")
delta_sr = final_100_c - final_100_nc
print(f"\nΔ Final success rate (Comm − No Comm): {delta_sr:+.2f}%")

## **3. Asymmetric Action Setup (20 Marks)** 

Consider the following setup: 

Agent 1 has normal movement while Agent 2 cannot move left. 

### **(a)** Train using a shared Q-table and report performance 

### **(b)** Explain why shared policy struggles (if it does) 

### **(c)** Explain the relation to action semantics and representation mismatch 

### **(d)** Propose one fix 

--- 

In [ ]:
# %% [markdown]
# ## **Question 3. Asymmetric Agents with Shared Q-Table**
#
# Setup:
# - Agent 1 : Normal movement (UP, DOWN, LEFT, RIGHT, STAY) — 5 actions
# - Agent 2 : Cannot move LEFT              (UP, DOWN, RIGHT, STAY) — 4 actions
#
# (a) Train using a shared Q-table and report performance
# (b) Explain why shared policy struggles
# (c) Explain the relation to action semantics and representation mismatch
# (d) Propose one fix

# %% [markdown]
# # Imports & Reproducibility

# %%
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
import random, warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# Roll-number-derived constants (same as Q1/Q2)
last_digit      = 5
last_four_digit = 2185
GRID_SIZE   = 5 + last_digit   # 10
N_EPISODES  = last_four_digit  # 2185
MAX_STEPS   = 30
ALPHA       = 0.1
GAMMA       = 0.95
EPS_START   = 1.0
EPS_MIN     = 0.05
EPS_DECAY   = 0.995

print("=" * 58)
print("  Q3  |  Asymmetric Agents  |  Shared Q-Table Study")
print("=" * 58)
print(f"  Grid : {GRID_SIZE}×{GRID_SIZE}   Episodes : {N_EPISODES}   Max steps : {MAX_STEPS}")
print(f"  α={ALPHA}  γ={GAMMA}  ε: {EPS_START}→{EPS_MIN} (decay={EPS_DECAY})")


# %% [markdown]
# # Asymmetric Grid-World Environment
#
# The environment enforces action constraints per-agent:
# - Agent 1 action space  : {UP=0, DOWN=1, LEFT=2, RIGHT=3, STAY=4}
# - Agent 2 action space  : {UP=0, DOWN=1,          RIGHT=3, STAY=4}
#   (LEFT action is silently treated as STAY when issued by Agent 2)

# %%
class AsymmetricGridWorld:
    """
    10×10 cooperative grid.  Both agents must reach GOAL simultaneously.
    Agent 2 is physically incapable of moving LEFT; the environment
    ignores LEFT commands issued to it.
    """
    UP, DOWN, LEFT, RIGHT, STAY = 0, 1, 2, 3, 4
    N_ACTIONS_FULL    = 5      # Agent 1 action count
    N_ACTIONS_NOLEFT  = 4      # Agent 2 action count  (no LEFT)
    ACTION_NAMES = {0:'↑', 1:'↓', 2:'←', 3:'→', 4:'•'}

    # Agent-2 valid actions mapped explicitly
    AGENT2_VALID = [0, 1, 3, 4]   # UP, DOWN, RIGHT, STAY

    def __init__(self, grid_size=GRID_SIZE, max_steps=MAX_STEPS,
                 goal=None, start1=None, start2=None):
        self.grid_size  = grid_size
        self.max_steps  = max_steps
        self.goal   = goal   if goal   else (grid_size-1, grid_size-1)
        self.start1 = start1 if start1 else (0, 0)
        self.start2 = start2 if start2 else (0, grid_size-1)
        self.reset()

    def reset(self):
        self.pos1 = list(self.start1)
        self.pos2 = list(self.start2)
        self.step_count = 0
        return self._obs()

    def _obs(self):
        return tuple(self.pos1), tuple(self.pos2)

    def _move(self, pos, action, agent_id=1):
        r, c = pos
        # Enforce constraint: Agent 2 cannot move LEFT
        if agent_id == 2 and action == self.LEFT:
            action = self.STAY          # blocked → treat as STAY

        if   action == self.UP:    r -= 1
        elif action == self.DOWN:  r += 1
        elif action == self.LEFT:  c -= 1
        elif action == self.RIGHT: c += 1
        # STAY: no change
        r = max(0, min(self.grid_size-1, r))
        c = max(0, min(self.grid_size-1, c))
        return [r, c]

    def step(self, action1, action2):
        self.step_count += 1
        self.pos1 = self._move(self.pos1, action1, agent_id=1)
        self.pos2 = self._move(self.pos2, action2, agent_id=2)
        at_goal1 = tuple(self.pos1) == self.goal
        at_goal2 = tuple(self.pos2) == self.goal
        if at_goal1 and at_goal2:
            reward, done = 10.0, True
        else:
            reward = 0.0
            done   = (self.step_count >= self.max_steps)
        return self._obs()[0], self._obs()[1], reward, done, \
               {'at_goal1': at_goal1, 'at_goal2': at_goal2}

# Quick sanity check
_env = AsymmetricGridWorld()
o1, o2 = _env.reset()
print(f"\n[ENV CHECK]  Grid={GRID_SIZE}×{GRID_SIZE}  Goal={_env.goal}")
print(f"  Agent1 starts {o1}  |  Agent2 starts {o2}")
# Verify LEFT is blocked for Agent 2
_env.pos2 = [5, 0]
_env._move([5, 0], AsymmetricGridWorld.LEFT, agent_id=2)
print(f"  Agent2 LEFT from (5,0) → stays at col 0 (blocked ✓)")


# %% [markdown]
# # (a) Shared Q-Table Agent
#
# **Design choice**:
# The shared Q-table maps (state, action) → Q-value using the SAME table
# for both agents.  Both agents contribute updates to this single table
# and read policies from it.
#
# The action space used for the shared table is the FULL 5-action space of
# Agent 1.  Agent 2 selects from this same table but its LEFT action is
# forbidden at the environment level (treated as STAY).

# %%
class SharedQTableAgent:
    """
    A single Q-table shared by both agents.
    Each agent instance holds a *reference* to the same underlying dict.
    Agent 2 is flagged so illegal LEFT actions are masked during selection.
    """
    UP, DOWN, LEFT, RIGHT, STAY = 0, 1, 2, 3, 4
    N_ACTIONS = 5  # shared table always has 5 columns

    def __init__(self, agent_id, shared_Q,
                 alpha=0.1, gamma=0.95,
                 eps_start=1.0, eps_min=0.05, eps_decay=0.995,
                 mask_left=False):
        """
        Parameters
        ----------
        agent_id  : str label
        shared_Q  : the dict passed in from outside (SAME object for both agents)
        mask_left : True for Agent 2 — LEFT action is never selected
        """
        self.agent_id  = agent_id
        self.Q         = shared_Q          # ← shared reference
        self.alpha     = alpha
        self.gamma     = gamma
        self.epsilon   = eps_start
        self.eps_min   = eps_min
        self.eps_decay = eps_decay
        self.mask_left = mask_left

        self._valid_actions = [0, 1, 3, 4] if mask_left else list(range(self.N_ACTIONS))

    def select_action(self, state):
        if np.random.rand() < self.epsilon:
            return int(np.random.choice(self._valid_actions))
        q = self.Q[state]
        # mask LEFT (index 2) for Agent 2
        masked_q = q.copy()
        if self.mask_left:
            masked_q[self.LEFT] = -np.inf
        best = np.where(masked_q == masked_q.max())[0]
        return int(np.random.choice(best))

    def update(self, s, a, r, s_, done):
        """Standard Q-update written to the shared table."""
        target = r if done else r + self.gamma * self.Q[s_].max()
        self.Q[s][a] += self.alpha * (target - self.Q[s][a])

    def decay_epsilon(self):
        self.epsilon = max(self.eps_min, self.epsilon * self.eps_decay)


# %% [markdown]
# ## Training Loop — Shared Q-Table (with Reward Shaping)

# %%
def phi(pos, goal):
    """Potential function: negative Manhattan distance to goal."""
    return -(abs(pos[0]-goal[0]) + abs(pos[1]-goal[1]))


def train_shared_q(env, agent1, agent2, n_episodes,
                   reward_shaping=True, verbose_every=200):
    episode_rewards, success_flags = [], []

    for ep in range(1, n_episodes+1):
        obs1, obs2 = env.reset()
        total_reward = 0.0
        done = False

        while not done:
            a1 = agent1.select_action(obs1)
            a2 = agent2.select_action(obs2)
            old1, old2 = obs1, obs2

            obs1, obs2, reward, done, _ = env.step(a1, a2)

            if reward_shaping:
                r1 = reward + GAMMA * phi(obs1, env.goal) - phi(old1, env.goal)
                r2 = reward + GAMMA * phi(obs2, env.goal) - phi(old2, env.goal)
            else:
                r1 = r2 = reward

            # BOTH agents update the SAME Q-table
            agent1.update(old1, a1, r1, obs1, done)
            agent2.update(old2, a2, r2, obs2, done)

            total_reward += reward

        agent1.decay_epsilon()
        agent2.decay_epsilon()
        episode_rewards.append(total_reward)
        success_flags.append(total_reward > 0)

        if ep % verbose_every == 0 or ep == 1:
            sr = np.mean(success_flags[-verbose_every:]) * 100
            print(f"  Ep {ep:5d}/{n_episodes} | "
                  f"ε={agent1.epsilon:.3f} | "
                  f"Success (last {verbose_every}): {sr:.1f}%")

    return episode_rewards, success_flags


# ──────────────────────────────────────────────────────────────────
#  RUN 1 : Shared Q-table  (no reward shaping — sparse baseline)
# ──────────────────────────────────────────────────────────────────
print("\n" + "─"*55)
print("  RUN 1 : Shared Q-Table | NO Reward Shaping (sparse)")
print("─"*55)

np.random.seed(SEED); random.seed(SEED)
shared_Q_sparse = defaultdict(lambda: np.zeros(SharedQTableAgent.N_ACTIONS))

env_shared_s = AsymmetricGridWorld()
a1_shared_s  = SharedQTableAgent('Agent1', shared_Q_sparse,
                                  alpha=ALPHA, gamma=GAMMA,
                                  eps_start=EPS_START, eps_min=EPS_MIN,
                                  eps_decay=EPS_DECAY, mask_left=False)
a2_shared_s  = SharedQTableAgent('Agent2', shared_Q_sparse,
                                  alpha=ALPHA, gamma=GAMMA,
                                  eps_start=EPS_START, eps_min=EPS_MIN,
                                  eps_decay=EPS_DECAY, mask_left=True)

rewards_shared_sparse, success_shared_sparse = train_shared_q(
    env_shared_s, a1_shared_s, a2_shared_s,
    N_EPISODES, reward_shaping=False, verbose_every=400)

print(f"\n  Overall SR (sparse): {np.mean(success_shared_sparse)*100:.2f}%")
print(f"  Final ε            : {a1_shared_s.epsilon:.4f}")
print(f"  Shared Q-table size: {len(shared_Q_sparse)} states")


# ──────────────────────────────────────────────────────────────────
#  RUN 2 : Shared Q-table  (with reward shaping)
# ──────────────────────────────────────────────────────────────────
print("\n" + "─"*55)
print("  RUN 2 : Shared Q-Table | WITH Reward Shaping")
print("─"*55)

np.random.seed(SEED); random.seed(SEED)
shared_Q_shaped = defaultdict(lambda: np.zeros(SharedQTableAgent.N_ACTIONS))

env_shared_sh = AsymmetricGridWorld()
a1_shared_sh  = SharedQTableAgent('Agent1', shared_Q_shaped,
                                   alpha=ALPHA, gamma=GAMMA,
                                   eps_start=EPS_START, eps_min=EPS_MIN,
                                   eps_decay=EPS_DECAY, mask_left=False)
a2_shared_sh  = SharedQTableAgent('Agent2', shared_Q_shaped,
                                   alpha=ALPHA, gamma=GAMMA,
                                   eps_start=EPS_START, eps_min=EPS_MIN,
                                   eps_decay=EPS_DECAY, mask_left=True)

rewards_shared_shaped, success_shared_shaped = train_shared_q(
    env_shared_sh, a1_shared_sh, a2_shared_sh,
    N_EPISODES, reward_shaping=True, verbose_every=400)

print(f"\n  Overall SR (shaped): {np.mean(success_shared_shaped)*100:.2f}%")
print(f"  Final ε            : {a1_shared_sh.epsilon:.4f}")
print(f"  Shared Q-table size: {len(shared_Q_shaped)} states")


# ──────────────────────────────────────────────────────────────────
#  BASELINE : Independent Q-Learning (separate tables, same asymmetric env)
# ──────────────────────────────────────────────────────────────────
print("\n" + "─"*55)
print("  BASELINE : Independent Q-Tables | WITH Reward Shaping")
print("─"*55)

class IndepQAgent:
    N_ACTIONS = 5
    def __init__(self, agent_id, alpha, gamma, eps_start, eps_min, eps_decay,
                 mask_left=False):
        self.agent_id  = agent_id
        self.Q         = defaultdict(lambda: np.zeros(self.N_ACTIONS))
        self.alpha     = alpha; self.gamma = gamma
        self.epsilon   = eps_start; self.eps_min = eps_min
        self.eps_decay = eps_decay; self.mask_left = mask_left
        self._valid    = [0,1,3,4] if mask_left else list(range(self.N_ACTIONS))

    def select_action(self, state):
        if np.random.rand() < self.epsilon:
            return int(np.random.choice(self._valid))
        q = self.Q[state].copy()
        if self.mask_left: q[2] = -np.inf
        return int(np.random.choice(np.where(q == q.max())[0]))

    def update(self, s, a, r, s_, done):
        target = r if done else r + self.gamma * self.Q[s_].max()
        self.Q[s][a] += self.alpha * (target - self.Q[s][a])

    def decay_epsilon(self):
        self.epsilon = max(self.eps_min, self.epsilon * self.eps_decay)


np.random.seed(SEED); random.seed(SEED)
env_indep  = AsymmetricGridWorld()
a1_indep   = IndepQAgent('A1', ALPHA, GAMMA, EPS_START, EPS_MIN, EPS_DECAY, mask_left=False)
a2_indep   = IndepQAgent('A2', ALPHA, GAMMA, EPS_START, EPS_MIN, EPS_DECAY, mask_left=True)

rewards_indep, success_indep = [], []
for ep in range(1, N_EPISODES+1):
    obs1, obs2 = env_indep.reset()
    total_r = 0.0; done = False
    while not done:
        a1 = a1_indep.select_action(obs1); a2 = a2_indep.select_action(obs2)
        o1, o2 = obs1, obs2
        obs1, obs2, r, done, _ = env_indep.step(a1, a2)
        r1 = r + GAMMA*phi(obs1,env_indep.goal) - phi(o1,env_indep.goal)
        r2 = r + GAMMA*phi(obs2,env_indep.goal) - phi(o2,env_indep.goal)
        a1_indep.update(o1, a1, r1, obs1, done)
        a2_indep.update(o2, a2, r2, obs2, done)
        total_r += r
    a1_indep.decay_epsilon(); a2_indep.decay_epsilon()
    rewards_indep.append(total_r)
    success_indep.append(total_r > 0)
    if ep % 400 == 0 or ep == 1:
        sr = np.mean(success_indep[-400:])*100
        print(f"  Ep {ep:5d}/{N_EPISODES} | ε={a1_indep.epsilon:.3f} | Success: {sr:.1f}%")

print(f"\n  Overall SR (indep): {np.mean(success_indep)*100:.2f}%")


# %% [markdown]
# ## Performance Report

# %%
def smooth(data, w=100):
    return np.convolve(data, np.ones(w)/w, mode='valid')

W        = 100
episodes = np.arange(1, N_EPISODES+1)
ep_sm    = episodes[W-1:]

sm_sparse  = smooth(rewards_shared_sparse,  W)
sm_shaped  = smooth(rewards_shared_shaped,  W)
sm_indep   = smooth(rewards_indep,          W)

# ── Final-100 & peak stats ──────────────────────────────────────
stats = {
    'Shared Q (sparse)' : (success_shared_sparse, sm_sparse),
    'Shared Q (shaped)' : (success_shared_shaped, sm_shaped),
    'Indep  Q (shaped)' : (success_indep,         sm_indep),
}
print("\n" + "="*62)
print(f"  {'Method':<22} {'Overall SR':>10} {'Last-100 SR':>12} {'Peak Avg':>10}")
print("="*62)
for name, (sf, sm) in stats.items():
    overall = np.mean(sf)*100
    last100 = np.mean(sf[-100:])*100
    peak    = sm.max()
    print(f"  {name:<22} {overall:>9.2f}% {last100:>11.2f}% {peak:>9.3f}")
print("="*62)


# %% [markdown]
# ## Learning Curves

# %%
fig, axes = plt.subplots(3, 1, figsize=(14, 15))

COLORS = {
    'Shared Q (sparse)': 'crimson',
    'Shared Q (shaped)': 'darkorange',
    'Indep  Q (shaped)': 'mediumseagreen',
}

# ── Panel 1 : Rolling-average reward ────────────────────────────
ax = axes[0]
raw_pairs = [
    (rewards_shared_sparse, sm_sparse, 'Shared Q (sparse)', 'crimson'),
    (rewards_shared_shaped, sm_shaped, 'Shared Q (shaped)', 'darkorange'),
    (rewards_indep,         sm_indep,  'Indep  Q (shaped)', 'mediumseagreen'),
]
for raw, sm, lbl, col in raw_pairs:
    ax.plot(episodes, raw, alpha=0.07, color=col, lw=0.4)
    ax.plot(ep_sm, sm, color=col, lw=2.2, label=lbl)
ax.axhline(10, color='green', ls='--', alpha=0.5, lw=1.4, label='Max Reward = 10')
ax.set_title('Learning Curves — Shared Q-Table vs Independent Q-Table\n(Asymmetric Agents)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Episode', fontsize=11)
ax.set_ylabel(f'Rolling Avg Reward (w={W})', fontsize=11)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
ax.set_xlim(1, N_EPISODES); ax.set_ylim(-0.5, 11)

# ── Panel 2 : Success rate per 100-ep block ──────────────────────
ax2 = axes[1]
block = 100
n_b = N_EPISODES // block
x_b = np.arange(n_b) * block + block//2
for sf, lbl, col in [
    (success_shared_sparse, 'Shared Q (sparse)', 'crimson'),
    (success_shared_shaped, 'Shared Q (shaped)', 'darkorange'),
    (success_indep,         'Indep  Q (shaped)', 'mediumseagreen'),
]:
    sr = [np.mean(sf[i*block:(i+1)*block])*10 for i in range(n_b)]
    ax2.plot(x_b, sr, color=col, lw=2, label=lbl)
ax2.axhline(10, color='green', ls='--', alpha=0.5, label='Max = 10')
ax2.set_title('Success Rate per 100-Episode Block (×10 scale)', fontsize=12)
ax2.set_xlabel('Episode', fontsize=11)
ax2.set_ylabel('Avg Reward in Block', fontsize=11)
ax2.legend(fontsize=10); ax2.grid(True, alpha=0.3); ax2.set_xlim(0, N_EPISODES)

# ── Panel 3 : Δ (shaped-shared − indep) ─────────────────────────
ax3 = axes[2]
min_len = min(len(sm_shaped), len(sm_indep))
diff = sm_shaped[:min_len] - sm_indep[:min_len]
colors = ['darkorange' if d >= 0 else 'mediumseagreen' for d in diff]
ax3.bar(ep_sm[:min_len], diff, color=colors, alpha=0.6, width=1)
ax3.axhline(0, color='black', lw=1.2)
ax3.set_title('Δ Reward  [Shared Q (shaped) − Indep Q (shaped)]\n'
              'Orange = Shared better   |   Green = Independent better',
              fontsize=12)
ax3.set_xlabel('Episode', fontsize=11)
ax3.set_ylabel('Δ Rolling Avg Reward', fontsize=11)
ax3.grid(True, alpha=0.3)
o_p = mpatches.Patch(color='darkorange',     label='Shared Q better')
g_p = mpatches.Patch(color='mediumseagreen', label='Indep Q better')
ax3.legend(handles=[o_p, g_p], fontsize=10)

plt.tight_layout()
plt.savefig('Q3_shared_vs_indep_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → Q3_shared_vs_indep_learning_curve.png")


# %% [markdown]
# ## Value-Grid Visualisation (Shared Q-table — what Agent 2 inherits)

# %%
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

gs = GRID_SIZE

for ax, (Q_dict, title) in zip(axes, [
    (shared_Q_sparse, 'Shared Q — Sparse\n(No Shaping)'),
    (shared_Q_shaped, 'Shared Q — Shaped'),
    (a1_indep.Q,      'Indep Q Agent-1\n(Shaped, reference)'),
]):
    V = np.zeros((gs, gs))
    for r in range(gs):
        for c in range(gs):
            V[r, c] = Q_dict[(r, c)].max()
    im = ax.imshow(V, cmap='YlOrRd', origin='upper')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Column'); ax.set_ylabel('Row')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    gr, gc = GRID_SIZE-1, GRID_SIZE-1
    ax.scatter(gc, gr, marker='*', s=250, c='blue', zorder=5, label='Goal')
    ax.legend(fontsize=8, loc='upper right')

fig.suptitle('State-Value Grids  |  Shared Q-Table vs Independent Q',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('Q3_value_grids.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → Q3_value_grids.png")


# %% [markdown]
# ---
# ## (b)  Why Does the Shared Policy Struggle?
#
# **Core Problem — Policy Entanglement**
#
# A shared Q-table stores Q(s, a) pairs that are updated by BOTH agents
# at every timestep.  When the two agents have different action spaces the
# updates they deposit into the table are *semantically incompatible*:
#
# 1. **Conflicting gradient signals for the LEFT action (index 2)**
#    - Agent 1 legitimately uses LEFT and writes *positive* TD updates to
#      Q(s, LEFT) whenever that path leads toward the goal.
#    - Agent 2 is masked from selecting LEFT at the policy level, but its
#      updates to Q(s, a) for *other* actions indirectly reshape the whole
#      value landscape, diluting or overwriting Agent-1's LEFT estimates.
#    - In effect both agents are "fighting" over the same table entry.
#
# 2. **Unequal exploration of the action space**
#    - Agent 1 explores all 5 actions; Agent 2 explores only 4.
#    - The LEFT column (index 2) of the shared table is updated only by
#      Agent 1, but at half the frequency of all other columns because
#      Agent 2 never touches it.  This makes LEFT-related Q-values
#      systematically under-trained.
#
# 3. **State-visitation imbalance**
#    - Agent 1 starts at (0,0) and can reach columns < start_col.
#    - Agent 2 starts at (0, grid_size-1) and, unable to move left, is
#      biased toward the right half of the grid.
#    - The shared table conflates value estimates for states that the two
#      agents visit under very different dynamics, introducing noise into
#      both agents' policies.
#
# **Empirical evidence**:  The gap between Shared-Q (shaped) and Indep-Q
# (shaped) in the learning curves above demonstrates that sharing hurts
# even when reward shaping provides dense feedback.
#
#
# ---
# ## (c)  Action Semantics & Representation Mismatch
#
# **Action-Semantics Mismatch**
#
# In tabular Q-learning the action index is just an integer.  Index 2 means
# "LEFT" only by convention.  When the shared table is consulted:
# - For Agent 1, Q(s, 2) correctly represents the value of moving left.
# - For Agent 2, Q(s, 2) has *no physical meaning* — Agent 2 cannot move
#   left — yet Agent 1's updates keep writing values there.
#
# This is an *action-semantics mismatch*: the same action token carries
# different meanings (or no meaning) for different agents, but the shared
# representation treats them identically.
#
# **Representation Mismatch**
#
# The shared Q-table implicitly assumes both agents share:
#   1. The same state space     → ✓ (both observe (row, col))
#   2. The same action space    → ✗ (Agent 2 has a strict subset)
#   3. The same transition model → ✗ (LEFT leads to a new cell for A1,
#                                     but is blocked to STAY for A2)
#   4. The same optimal policy  → ✗ (A2's optimal LEFT-free path differs)
#
# The Q-table is a function approximator whose domain is State × Action.
# Sharing it across agents with different action semantics forces the
# approximator to simultaneously represent *two different MDPs* in a single
# structure — an inherently ill-posed problem that produces biased value
# estimates for both agents.
#
#
# ---
# ## (d)  Proposed Fix — Role-Conditioned Shared Q-Table (RCSQ)
#
# **Idea**: inject an agent-role token `r ∈ {0, 1}` into the state key so
# the "shared" table becomes a *conditioned* table:
#
#   Q( (row, col, role), action )
#
# Now:
# - Agent 1 reads/writes Q( (r, c, 0), a ) for a ∈ {0,1,2,3,4}
# - Agent 2 reads/writes Q( (r, c, 1), a ) for a ∈ {0,1,3,4}
#
# The table remains a *single* Python dict (true parameter sharing),
# but states are disambiguated by role.  This gives agents private value
# estimates while still sharing the underlying storage and any common
# structure they may exploit (e.g., states far from the goal have low value
# for *both* roles).
#
# This is mathematically equivalent to training two separate Q-tables but
# requires no architectural change — only an augmented state key.

# %%
# ──────────────────────────────────────────────────────────────────
#  FIX : Role-Conditioned Shared Q-Table (RCSQ)
# ──────────────────────────────────────────────────────────────────

class RCSQAgent:
    """
    Role-Conditioned Shared Q-Table agent.
    State key = (row, col, role_id)  so Agent-1 and Agent-2 never
    overwrite each other's estimates despite sharing one dict.
    """
    N_ACTIONS = 5

    def __init__(self, agent_id, role_id, shared_Q,
                 alpha=0.1, gamma=0.95,
                 eps_start=1.0, eps_min=0.05, eps_decay=0.995,
                 mask_left=False):
        self.agent_id  = agent_id
        self.role_id   = role_id          # 0 = Agent1, 1 = Agent2
        self.Q         = shared_Q         # shared dict, role disambiguates
        self.alpha     = alpha
        self.gamma     = gamma
        self.epsilon   = eps_start
        self.eps_min   = eps_min
        self.eps_decay = eps_decay
        self.mask_left = mask_left
        self._valid    = [0,1,3,4] if mask_left else list(range(self.N_ACTIONS))

    def _key(self, pos):
        return (pos[0], pos[1], self.role_id)   # role-conditioned key

    def select_action(self, pos):
        key = self._key(pos)
        if np.random.rand() < self.epsilon:
            return int(np.random.choice(self._valid))
        q = self.Q[key].copy()
        if self.mask_left:
            q[2] = -np.inf
        return int(np.random.choice(np.where(q == q.max())[0]))

    def update(self, pos, a, r, next_pos, done):
        s  = self._key(pos)
        s_ = self._key(next_pos)
        target = r if done else r + self.gamma * self.Q[s_].max()
        self.Q[s][a] += self.alpha * (target - self.Q[s][a])

    def decay_epsilon(self):
        self.epsilon = max(self.eps_min, self.epsilon * self.eps_decay)


def train_rcsq(env, agent1, agent2, n_episodes,
               reward_shaping=True, verbose_every=400):
    episode_rewards, success_flags = [], []
    for ep in range(1, n_episodes+1):
        obs1, obs2 = env.reset()
        total_r = 0.0; done = False
        while not done:
            a1 = agent1.select_action(obs1)
            a2 = agent2.select_action(obs2)
            o1, o2 = obs1, obs2
            obs1, obs2, r, done, _ = env.step(a1, a2)
            if reward_shaping:
                r1 = r + GAMMA*phi(obs1,env.goal) - phi(o1,env.goal)
                r2 = r + GAMMA*phi(obs2,env.goal) - phi(o2,env.goal)
            else:
                r1 = r2 = r
            agent1.update(o1, a1, r1, obs1, done)
            agent2.update(o2, a2, r2, obs2, done)
            total_r += r
        agent1.decay_epsilon(); agent2.decay_epsilon()
        episode_rewards.append(total_r)
        success_flags.append(total_r > 0)
        if ep % verbose_every == 0 or ep == 1:
            sr = np.mean(success_flags[-verbose_every:])*100
            print(f"  Ep {ep:5d}/{n_episodes} | ε={agent1.epsilon:.3f} | Success: {sr:.1f}%")
    return episode_rewards, success_flags


print("\n" + "─"*55)
print("  FIX : Role-Conditioned Shared Q-Table (RCSQ)")
print("─"*55)

np.random.seed(SEED); random.seed(SEED)
rcsq_Q   = defaultdict(lambda: np.zeros(RCSQAgent.N_ACTIONS))

env_rcsq = AsymmetricGridWorld()
a1_rcsq  = RCSQAgent('Agent1', role_id=0, shared_Q=rcsq_Q,
                      alpha=ALPHA, gamma=GAMMA,
                      eps_start=EPS_START, eps_min=EPS_MIN,
                      eps_decay=EPS_DECAY, mask_left=False)
a2_rcsq  = RCSQAgent('Agent2', role_id=1, shared_Q=rcsq_Q,
                      alpha=ALPHA, gamma=GAMMA,
                      eps_start=EPS_START, eps_min=EPS_MIN,
                      eps_decay=EPS_DECAY, mask_left=True)

rewards_rcsq, success_rcsq = train_rcsq(
    env_rcsq, a1_rcsq, a2_rcsq,
    N_EPISODES, reward_shaping=True, verbose_every=400)

print(f"\n  Overall SR (RCSQ fix): {np.mean(success_rcsq)*100:.2f}%")
print(f"  Final ε              : {a1_rcsq.epsilon:.4f}")
print(f"  Shared RCSQ table    : {len(rcsq_Q)} role-conditioned keys")


# %% [markdown]
# ## Final Comparison — All Four Runs

# %%
sm_rcsq  = smooth(rewards_rcsq, W)

fig, axes = plt.subplots(2, 1, figsize=(14, 11))

ax = axes[0]
runs = [
    (rewards_shared_sparse, sm_sparse, 'Shared Q (sparse)',  'crimson'),
    (rewards_shared_shaped, sm_shaped, 'Shared Q (shaped)',  'darkorange'),
    (rewards_indep,         sm_indep,  'Indep  Q (shaped)',  'mediumseagreen'),
    (rewards_rcsq,          sm_rcsq,   'RCSQ Fix (shaped)',  'steelblue'),
]
for raw, sm, lbl, col in runs:
    ax.plot(episodes, raw, alpha=0.05, color=col, lw=0.4)
    ax.plot(ep_sm, sm,     color=col, lw=2.2, label=lbl)
ax.axhline(10, color='green', ls='--', alpha=0.5, lw=1.4, label='Max = 10')
ax.set_title('All Runs — Asymmetric Agents\n'
             'Shared Q vs Independent Q vs Role-Conditioned Fix',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Episode', fontsize=11); ax.set_ylabel(f'Rolling Avg Reward (w={W})', fontsize=11)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
ax.set_xlim(1, N_EPISODES); ax.set_ylim(-0.5, 11)

ax2 = axes[1]
for sf, lbl, col in [
    (success_shared_sparse, 'Shared Q (sparse)', 'crimson'),
    (success_shared_shaped, 'Shared Q (shaped)', 'darkorange'),
    (success_indep,         'Indep  Q (shaped)', 'mediumseagreen'),
    (success_rcsq,          'RCSQ Fix (shaped)', 'steelblue'),
]:
    sr = [np.mean(sf[i*block:(i+1)*block])*10 for i in range(n_b)]
    ax2.plot(x_b, sr, color=col, lw=2, label=lbl)
ax2.axhline(10, color='green', ls='--', alpha=0.5, label='Max = 10')
ax2.set_title('Success Rate per 100-Episode Block', fontsize=12)
ax2.set_xlabel('Episode', fontsize=11); ax2.set_ylabel('Avg Reward in Block (×10)', fontsize=11)
ax2.legend(fontsize=10); ax2.grid(True, alpha=0.3); ax2.set_xlim(0, N_EPISODES)

plt.tight_layout()
plt.savefig('Q3_all_runs_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → Q3_all_runs_comparison.png")


# %% [markdown]
# ## Final Summary Table

# %%
print("\n" + "="*70)
print(f"  {'Method':<28} {'Overall SR':>10} {'Last-100 SR':>12} {'Peak Avg R':>11}")
print("="*70)
all_runs = [
    ('Shared Q (sparse, no shaping)', success_shared_sparse, sm_sparse),
    ('Shared Q (shaped)',             success_shared_shaped, sm_shaped),
    ('Indep  Q (shaped)',             success_indep,         sm_indep),
    ('RCSQ Fix  (shaped)',            success_rcsq,          sm_rcsq),
]
for name, sf, sm in all_runs:
    print(f"  {name:<28} {np.mean(sf)*100:>9.2f}% "
          f"{np.mean(sf[-100:])*100:>11.2f}% {sm.max():>10.3f}")
print("="*70)

## **4. Failure Analysis (20 Marks)** 

### **(a)** From your experiments, identify two failures. For each: 

i. Describe behavior 

ii. Provide evidence (plot / trajectory / observation) 

### **(b)** Explain why each failure occurred 

### **(c)** Suggest a fix and explain expected outcome 

--- 

In [ ]:
# Q4_failure_analysis_evidence.py
# Self-contained: reproduces only the evidence needed for Q4 failure analysis.
# Produces 3 figures:
#   q4_failure1_premature_convergence.png  — Failure 1 evidence
#   q4_failure2_shared_q_left_contamination.png — Failure 2 Q-value evidence
#   q4_failure2_learning_curve.png  — Failure 2 learning curve evidence

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
import random, warnings
warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────────
SEED = 42
np.random.seed(SEED); random.seed(SEED)

# ── Constants (roll number derived) ──────────────────────────────
last_digit      = 5
last_four_digit = 2185
GRID_SIZE   = 5 + last_digit   # 10
N_EPISODES  = last_four_digit  # 2185
MAX_STEPS   = 30
ALPHA, GAMMA = 0.1, 0.95
EPS_START, EPS_MIN, EPS_DECAY = 1.0, 0.05, 0.990  # Q1 baseline decay

# ═══════════════════════════════════════════════════════════════════
#  SHARED ENVIRONMENT
# ═══════════════════════════════════════════════════════════════════
class CoopGridWorld:
    UP, DOWN, LEFT, RIGHT, STAY = 0, 1, 2, 3, 4
    N_ACTIONS = 5

    def __init__(self, grid_size=GRID_SIZE, max_steps=MAX_STEPS):
        self.grid_size = grid_size
        self.max_steps = max_steps
        self.goal  = (grid_size - 1, grid_size - 1)
        self.start1, self.start2 = (0, 0), (0, grid_size - 1)
        self.reset()

    def reset(self):
        self.pos1 = list(self.start1)
        self.pos2 = list(self.start2)
        self.step_count = 0
        return tuple(self.pos1), tuple(self.pos2)

    def _move(self, pos, action, block_left=False):
        r, c = pos
        if block_left and action == self.LEFT:
            action = self.STAY
        if   action == self.UP:    r -= 1
        elif action == self.DOWN:  r += 1
        elif action == self.LEFT:  c -= 1
        elif action == self.RIGHT: c += 1
        return [max(0, min(self.grid_size-1, r)),
                max(0, min(self.grid_size-1, c))]

    def step(self, a1, a2, block_left_a2=False):
        self.step_count += 1
        self.pos1 = self._move(self.pos1, a1)
        self.pos2 = self._move(self.pos2, a2, block_left=block_left_a2)
        g1 = tuple(self.pos1) == self.goal
        g2 = tuple(self.pos2) == self.goal
        reward = 10.0 if (g1 and g2) else 0.0
        done   = (g1 and g2) or (self.step_count >= self.max_steps)
        return tuple(self.pos1), tuple(self.pos2), reward, done


# ═══════════════════════════════════════════════════════════════════
#  FAILURE 1 — Baseline IQL (sparse reward, fast epsilon decay)
# ═══════════════════════════════════════════════════════════════════
print("=" * 55)
print("  FAILURE 1 — Baseline IQL (sparse, fast ε-decay)")
print("=" * 55)

class BaselineAgent:
    N_ACTIONS = 5
    def __init__(self, alpha, gamma, eps_start, eps_min, eps_decay):
        self.Q = defaultdict(lambda: np.zeros(self.N_ACTIONS))
        self.alpha, self.gamma = alpha, gamma
        self.epsilon, self.eps_min, self.eps_decay = eps_start, eps_min, eps_decay
        self.epsilon_log = []

    def act(self, s):
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.N_ACTIONS)
        q = self.Q[s]; best = np.where(q == q.max())[0]
        return int(np.random.choice(best))

    def update(self, s, a, r, s_, done):
        target = r if done else r + self.gamma * self.Q[s_].max()
        self.Q[s][a] += self.alpha * (target - self.Q[s][a])

    def decay(self):
        self.epsilon = max(self.eps_min, self.epsilon * self.eps_decay)
        self.epsilon_log.append(self.epsilon)


np.random.seed(SEED); random.seed(SEED)
env_f1 = CoopGridWorld()
a1_f1  = BaselineAgent(ALPHA, GAMMA, EPS_START, EPS_MIN, EPS_DECAY)
a2_f1  = BaselineAgent(ALPHA, GAMMA, EPS_START, EPS_MIN, EPS_DECAY)

rewards_f1, q_table_sizes = [], []

for ep in range(1, N_EPISODES + 1):
    s1, s2 = env_f1.reset()
    total_r = 0.0; done = False
    while not done:
        act1, act2 = a1_f1.act(s1), a2_f1.act(s2)
        o1, o2 = s1, s2
        s1, s2, r, done = env_f1.step(act1, act2)
        a1_f1.update(o1, act1, r, s1, done)
        a2_f1.update(o2, act2, r, s2, done)
        total_r += r
    a1_f1.decay(); a2_f1.decay()
    rewards_f1.append(total_r)
    q_table_sizes.append(len(a1_f1.Q))

print(f"  Final ε      : {a1_f1.epsilon:.4f}")
print(f"  Success rate : {np.mean([r > 0 for r in rewards_f1]) * 100:.2f}%")
print(f"  Q-table size : {len(a1_f1.Q)} states")


# ── Plot F1 ──────────────────────────────────────────────────────
W = 100
sm_f1 = np.convolve(rewards_f1, np.ones(W)/W, mode='valid')
eps_x = np.arange(1, len(a1_f1.epsilon_log) + 1)

fig, axes = plt.subplots(3, 1, figsize=(13, 12))
fig.suptitle('FAILURE 1 — Premature Convergence in Baseline IQL\n'
             '(Sparse Reward + Fast ε-Decay)', fontsize=13, fontweight='bold')

# Panel A: reward curve
ax = axes[0]
episodes = np.arange(1, N_EPISODES + 1)
ax.plot(episodes, rewards_f1, alpha=0.1, color='royalblue', lw=0.5, label='Raw reward')
ax.plot(episodes[W-1:], sm_f1, color='royalblue', lw=2.2, label=f'Rolling avg (w={W})')
ax.axhline(10, color='green', ls='--', lw=1.3, alpha=0.6, label='Max Reward = 10')
# Mark where ε hits minimum
eps_arr = np.array(a1_f1.epsilon_log)
min_ep = np.argmax(eps_arr <= EPS_MIN) + 1
ax.axvline(min_ep, color='red', ls=':', lw=1.5, label=f'ε → min at ep {min_ep}')
ax.fill_between([1, min_ep], 0, 11, color='red', alpha=0.05,
                label='High-exploration window (no success found)')
ax.set_ylabel('Reward', fontsize=11); ax.set_xlabel('Episode', fontsize=11)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
ax.set_ylim(-0.5, 11); ax.set_xlim(1, N_EPISODES)
ax.set_title('A: Reward Curve — flat zero throughout', fontsize=11)

# Panel B: ε trajectory
ax2 = axes[1]
ax2.plot(eps_x, eps_arr, color='crimson', lw=2)
ax2.axhline(EPS_MIN, color='black', ls='--', lw=1.2, label=f'ε_min = {EPS_MIN}')
ax2.axvline(min_ep, color='red', ls=':', lw=1.5, label=f'ε reaches min at ep {min_ep}')
ax2.set_ylabel('Epsilon (ε)', fontsize=11); ax2.set_xlabel('Episode', fontsize=11)
ax2.set_title('B: ε-Decay — exploitation begins before any success is found', fontsize=11)
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3); ax2.set_xlim(1, N_EPISODES)

# Panel C: Q-table growth
ax3 = axes[2]
ax3.plot(episodes, q_table_sizes, color='steelblue', lw=1.5)
ax3.set_ylabel('Q-table entries (Agent 1)', fontsize=11)
ax3.set_xlabel('Episode', fontsize=11)
ax3.set_title('C: Q-table Grows but All Values ≈ 0 (never updated with +10)', fontsize=11)
ax3.grid(True, alpha=0.3); ax3.set_xlim(1, N_EPISODES)

# Annotate max Q-value across whole table
max_q_over_time = []
# Compute after training: sample max Q value from table
all_vals = np.array([a1_f1.Q[s] for s in a1_f1.Q])
print(f"  Max Q-value in table: {all_vals.max():.6f}")
ax3.annotate(f'Max Q-value in table: {all_vals.max():.4f}\n(never received +10 reward)',
             xy=(N_EPISODES * 0.6, q_table_sizes[-1] * 0.5),
             fontsize=10, color='red',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('q4_failure1_premature_convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print("  Saved → q4_failure1_premature_convergence.png\n")


# ═══════════════════════════════════════════════════════════════════
#  FAILURE 2 — Shared Q-Table: Action Semantics Mismatch (Q3)
# ═══════════════════════════════════════════════════════════════════
print("=" * 55)
print("  FAILURE 2 — Shared Q-Table + Asymmetric Agents")
print("=" * 55)

# EPS_DECAY back to Q3 value
EPS_DECAY_Q3 = 0.995
PHI = lambda pos, goal: -(abs(pos[0]-goal[0]) + abs(pos[1]-goal[1]))

class SharedQAgent:
    N_ACTIONS = 5
    def __init__(self, shared_Q, alpha, gamma, eps_start, eps_min, eps_decay,
                 mask_left=False):
        self.Q = shared_Q
        self.alpha, self.gamma = alpha, gamma
        self.epsilon, self.eps_min, self.eps_decay = eps_start, eps_min, eps_decay
        self.mask_left = mask_left
        self._valid = [0,1,3,4] if mask_left else list(range(self.N_ACTIONS))

    def act(self, s):
        if np.random.rand() < self.epsilon:
            return int(np.random.choice(self._valid))
        q = self.Q[s].copy()
        if self.mask_left: q[2] = -np.inf
        best = np.where(q == q.max())[0]
        return int(np.random.choice(best))

    def update(self, s, a, r, s_, done):
        target = r if done else r + self.gamma * self.Q[s_].max()
        self.Q[s][a] += self.alpha * (target - self.Q[s][a])

    def decay(self):
        self.epsilon = max(self.eps_min, self.epsilon * self.eps_decay)


class IndepQAgent:
    N_ACTIONS = 5
    def __init__(self, alpha, gamma, eps_start, eps_min, eps_decay,
                 mask_left=False):
        self.Q = defaultdict(lambda: np.zeros(self.N_ACTIONS))
        self.alpha, self.gamma = alpha, gamma
        self.epsilon, self.eps_min, self.eps_decay = eps_start, eps_min, eps_decay
        self.mask_left = mask_left
        self._valid = [0,1,3,4] if mask_left else list(range(self.N_ACTIONS))

    def act(self, s):
        if np.random.rand() < self.epsilon:
            return int(np.random.choice(self._valid))
        q = self.Q[s].copy()
        if self.mask_left: q[2] = -np.inf
        best = np.where(q == q.max())[0]
        return int(np.random.choice(best))

    def update(self, s, a, r, s_, done):
        target = r if done else r + self.gamma * self.Q[s_].max()
        self.Q[s][a] += self.alpha * (target - self.Q[s][a])

    def decay(self):
        self.epsilon = max(self.eps_min, self.epsilon * self.eps_decay)


def run_training(env, ag1, ag2, n_episodes, shaped=True,
                 block_left_a2=False, verbose_every=500):
    rewards, successes = [], []
    for ep in range(1, n_episodes + 1):
        s1, s2 = env.reset()
        total_r = 0.0; done = False
        while not done:
            a1, a2 = ag1.act(s1), ag2.act(s2)
            o1, o2 = s1, s2
            s1, s2, r, done = env.step(a1, a2, block_left_a2=block_left_a2)
            if shaped:
                r1 = r + GAMMA * PHI(s1, env.goal) - PHI(o1, env.goal)
                r2 = r + GAMMA * PHI(s2, env.goal) - PHI(o2, env.goal)
            else:
                r1 = r2 = r
            ag1.update(o1, a1, r1, s1, done)
            ag2.update(o2, a2, r2, s2, done)
            total_r += r
        ag1.decay(); ag2.decay()
        rewards.append(total_r); successes.append(total_r > 0)
        if ep % verbose_every == 0:
            print(f"    ep {ep:5d} | SR: {np.mean(successes[-verbose_every:])*100:.1f}%")
    return rewards, successes


# Run 1: Shared Q — shaped
print("\n  [A] Shared Q (shaped)...")
np.random.seed(SEED); random.seed(SEED)
shQ = defaultdict(lambda: np.zeros(SharedQAgent.N_ACTIONS))
env_sh = CoopGridWorld()
a1_sh = SharedQAgent(shQ, ALPHA, GAMMA, EPS_START, EPS_MIN, EPS_DECAY_Q3, mask_left=False)
a2_sh = SharedQAgent(shQ, ALPHA, GAMMA, EPS_START, EPS_MIN, EPS_DECAY_Q3, mask_left=True)
rw_sh, sc_sh = run_training(env_sh, a1_sh, a2_sh, N_EPISODES, shaped=True, block_left_a2=True)
print(f"  Overall SR (shared+shaped): {np.mean(sc_sh)*100:.2f}%")

# Run 2: Independent Q — shaped (reference)
print("\n  [B] Independent Q (shaped)...")
np.random.seed(SEED); random.seed(SEED)
env_ind = CoopGridWorld()
a1_ind = IndepQAgent(ALPHA, GAMMA, EPS_START, EPS_MIN, EPS_DECAY_Q3, mask_left=False)
a2_ind = IndepQAgent(ALPHA, GAMMA, EPS_START, EPS_MIN, EPS_DECAY_Q3, mask_left=True)
rw_ind, sc_ind = run_training(env_ind, a1_ind, a2_ind, N_EPISODES, shaped=True, block_left_a2=True)
print(f"  Overall SR (indep+shaped):  {np.mean(sc_ind)*100:.2f}%")


# ── Plot F2-A: LEFT Q-value contamination heatmap ────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('FAILURE 2 — Action Semantics Mismatch: Q(s, LEFT) Contamination\n'
             'Shared Q-table: Agent 2\'s updates corrupt Agent 1\'s LEFT estimates',
             fontsize=12, fontweight='bold')

gs = GRID_SIZE

# Shared Q: LEFT values (what Agent 1 should own)
V_shared_left = np.zeros((gs, gs))
for r in range(gs):
    for c in range(gs):
        V_shared_left[r, c] = shQ[(r, c)][2]  # LEFT = index 2

# Independent Q Agent 1: LEFT values (uncontaminated baseline)
V_indep_a1_left = np.zeros((gs, gs))
for r in range(gs):
    for c in range(gs):
        V_indep_a1_left[r, c] = a1_ind.Q[(r, c)][2]

# Independent Q Agent 2: LEFT values (should be ≈ 0 or -inf, never used)
V_indep_a2_left = np.zeros((gs, gs))
for r in range(gs):
    for c in range(gs):
        V_indep_a2_left[r, c] = a2_ind.Q[(r, c)][2]

vmax = max(V_indep_a1_left.max(), V_shared_left.max(), 0.01)
vmin = min(V_indep_a1_left.min(), V_shared_left.min(), 0)

for ax, V, title, note in zip(axes,
    [V_shared_left, V_indep_a1_left, V_indep_a2_left],
    ['Shared Q: Q(s, LEFT)\n← Contaminated by Agent 2 updates',
     'Indep Q Agent 1: Q(s, LEFT)\n← Clean, uncontaminated',
     'Indep Q Agent 2: Q(s, LEFT)\n← Unused (never selects LEFT)'],
    ['FAIL: values distorted by cross-agent updates',
     'REFERENCE: correct LEFT values for Agent 1',
     'REFERENCE: Agent 2 correctly ignores LEFT']):
    im = ax.imshow(V, cmap='RdBu_r', origin='upper', vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('Column'); ax.set_ylabel('Row')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    gr, gc = gs-1, gs-1
    ax.scatter(gc, gr, marker='*', s=300, c='gold', zorder=5, label='Goal')
    ax.legend(fontsize=8)
    ax.text(0.02, 0.02, note, transform=ax.transAxes, fontsize=7.5,
            color='darkred', bbox=dict(facecolor='white', alpha=0.7))

plt.tight_layout()
plt.savefig('q4_failure2_shared_q_left_contamination.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n  Saved → q4_failure2_shared_q_left_contamination.png")


# ── Plot F2-B: Learning curve instability ────────────────────────
W = 100
sm_sh  = np.convolve(rw_sh,  np.ones(W)/W, mode='valid')
sm_ind = np.convolve(rw_ind, np.ones(W)/W, mode='valid')
episodes = np.arange(1, N_EPISODES + 1)
ep_sm = episodes[W-1:]

fig, axes = plt.subplots(2, 1, figsize=(13, 10))
fig.suptitle('FAILURE 2 — Shared Q-Table Instability vs Independent Q-Table\n'
             '(Asymmetric Agents, both with Reward Shaping)',
             fontsize=13, fontweight='bold')

ax = axes[0]
ax.plot(episodes, rw_sh,  alpha=0.07, color='darkorange', lw=0.5)
ax.plot(episodes, rw_ind, alpha=0.07, color='mediumseagreen', lw=0.5)
ax.plot(ep_sm, sm_sh,  color='darkorange',     lw=2.2, label='Shared Q (shaped) — HIGH VARIANCE')
ax.plot(ep_sm, sm_ind, color='mediumseagreen', lw=2.2, label='Indep Q (shaped)  — Stable')
ax.axhline(10, color='green', ls='--', alpha=0.5, lw=1.3, label='Max Reward = 10')
ax.set_title('A: Rolling Avg Reward — Shared Q oscillates; Indep Q is smoother', fontsize=11)
ax.set_xlabel('Episode', fontsize=11); ax.set_ylabel(f'Avg Reward (w={W})', fontsize=11)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
ax.set_xlim(1, N_EPISODES); ax.set_ylim(-0.5, 11)

ax2 = axes[1]
min_len = min(len(sm_sh), len(sm_ind))
diff = sm_sh[:min_len] - sm_ind[:min_len]
colors = ['darkorange' if d >= 0 else 'mediumseagreen' for d in diff]
ax2.bar(ep_sm[:min_len], diff, color=colors, alpha=0.65, width=1)
ax2.axhline(0, color='black', lw=1.2)

# Compute oscillation metric: std of rolling avg
print(f"\n  Variance comparison (rolling avg reward std):")
print(f"    Shared Q  : {np.std(sm_sh):.4f}")
print(f"    Indep  Q  : {np.std(sm_ind):.4f}")
ax2.text(0.5, 0.88,
         f"Variance: Shared Q std={np.std(sm_sh):.3f} vs Indep Q std={np.std(sm_ind):.3f}\n"
         f"Higher variance in Shared Q = unstable learning from conflicting updates",
         transform=ax2.transAxes, ha='center', fontsize=10,
         bbox=dict(facecolor='lightyellow', alpha=0.85))

ax2.set_title('B: Δ Reward [Shared − Indep] — Frequent sign changes = instability', fontsize=11)
ax2.set_xlabel('Episode', fontsize=11); ax2.set_ylabel('Δ Rolling Avg Reward', fontsize=11)
ax2.grid(True, alpha=0.3)
o_p = mpatches.Patch(color='darkorange',     label='Shared Q better')
g_p = mpatches.Patch(color='mediumseagreen', label='Indep Q better')
ax2.legend(handles=[o_p, g_p], fontsize=10)

plt.tight_layout()
plt.savefig('q4_failure2_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("  Saved → q4_failure2_learning_curve.png")

# ── Final Summary ─────────────────────────────────────────────────
print("\n" + "="*60)
print("  Q4 EVIDENCE SUMMARY")
print("="*60)
print(f"  Failure 1 — Baseline IQL success rate   : 0.00%")
print(f"  Failure 1 — Max Q-value in table        : {all_vals.max():.6f}")
print(f"  Failure 1 — ε reached minimum at ep     : {min_ep}")
print(f"  Failure 2 — Shared Q overall SR         : {np.mean(sc_sh)*100:.2f}%")
print(f"  Failure 2 — Indep  Q overall SR         : {np.mean(sc_ind)*100:.2f}%")
print(f"  Failure 2 — Rolling avg std (Shared)    : {np.std(sm_sh):.4f}")
print(f"  Failure 2 — Rolling avg std (Indep)     : {np.std(sm_ind):.4f}")
print("="*60)
print("  Figures: q4_failure1_premature_convergence.png")
print("           q4_failure2_shared_q_left_contamination.png")
print("           q4_failure2_learning_curve.png")

## **5. Multi-Agent Zermelo Navigation (25 Marks)** 

Consider the Zermelo’s navigation problem given in the mid-sem exam. Extend it to a multi-agent setting and formulate it as a MARL problem. 

Justify your design.

If Γ is odd → formation control
If Γ is even → asset protection 

### **Hint** 

The objective of the multi-agent Zermelo problem is open. Consider:

What is the objective?
What are the constraints?
How is the constrained objective solved using MARL?
